In [ ]:
!pip install -q transformers peft accelerate trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.1 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM ,AutoTokenizer,BitsAndBytesConfig
from peft import LoraConfig, get_peft_model,prepare_model_for_kbit_training
from datasets import load_dataset,concatenate_datasets
from trl import SFTTrainer,SFTConfig

In [ ]:
languages=['python','javascript']

In [ ]:
Train_datasets=[load_dataset('code-search-net/code_search_net',lang,split="train[:2500]") for lang in languages ]
train_dataset=concatenate_datasets(Train_datasets)
Val_datasets=[load_dataset('code-search-net/code_search_net',lang,split="validation[:500]") for lang in languages ]
val_dataset=concatenate_datasets(Val_datasets)
Test_datasets=[load_dataset('code-search-net/code_search_net',lang,split="test[:500]") for lang in languages ]
test_dataset=concatenate_datasets(Test_datasets)

README.md:   0%|          | 0.00/14.1k [00:00<?, ?B/s]

python/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  522MB            

python/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

python/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 28.7MB            

python/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

javascript/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 10.3MB            

javascript/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/123889 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6483 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8253 [00:00<?, ? examples/s]

In [ ]:
train_dataset.shape

(5000, 11)

In [ ]:
val_dataset.shape

(1000, 11)

In [ ]:
test_dataset.shape

(1000, 11)

In [ ]:
train_dataset.column_names

['repository_name',
 'func_path_in_repository',
 'func_name',
 'whole_func_string',
 'language',
 'func_code_string',
 'func_code_tokens',
 'func_documentation_string',
 'func_documentation_tokens',
 'split_name',
 'func_code_url']

In [ ]:
train_dataset[0]['func_documentation_string']

'Estimate discontinuity in basis of low resolution image segmentation.\n        :return: discontinuity in low resolution'

In [ ]:
import re
def remove_docstring(code):
  pattern=r'("""["\s\S]*?"""|\'\'\'["\s\S]*?\'\'\')'
  return re.sub(pattern, '', code,count=1).strip()

In [ ]:
def format_example(example):
  code_without_docstring=remove_docstring(example['func_code_string'])
  example['text']=f"[INST] Write a docstring for this function:\n{code_without_docstring} [/INST] {example['func_documentation_string']}"
  return example

In [ ]:
train_dataset=train_dataset.map(format_example)
val_dataset=val_dataset.map(format_example)
test_dataset=test_dataset.map(format_example)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
train_dataset.column_names

['repository_name',
 'func_path_in_repository',
 'func_name',
 'whole_func_string',
 'language',
 'func_code_string',
 'func_code_tokens',
 'func_documentation_string',
 'func_documentation_tokens',
 'split_name',
 'func_code_url',
 'text']

In [ ]:
val_dataset.column_names

['repository_name',
 'func_path_in_repository',
 'func_name',
 'whole_func_string',
 'language',
 'func_code_string',
 'func_code_tokens',
 'func_documentation_string',
 'func_documentation_tokens',
 'split_name',
 'func_code_url',
 'text']

In [ ]:
test_dataset.column_names

['repository_name',
 'func_path_in_repository',
 'func_name',
 'whole_func_string',
 'language',
 'func_code_string',
 'func_code_tokens',
 'func_documentation_string',
 'func_documentation_tokens',
 'split_name',
 'func_code_url',
 'text']

In [ ]:
train_dataset[0]['text']

'[INST] Write a docstring for this function:\ndef __msgc_step3_discontinuity_localization(self):\n        \n        import scipy\n\n        start = self._start_time\n        seg = 1 - self.segmentation.astype(np.int8)\n        self.stats["low level object voxels"] = np.sum(seg)\n        self.stats["low level image voxels"] = np.prod(seg.shape)\n        # in seg is now stored low resolution segmentation\n        # back to normal parameters\n        # step 2: discontinuity localization\n        # self.segparams = sparams_hi\n        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")\n        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))\n        # logger.debug(str(np.max(seg_border)))\n        # logger.debug(str(np.min(seg_border)))\n        seg_border[seg_border != 0] = 1\n        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))\n        # scipy.ndimage.morphology.distance_transform_edt\n        boundary_dilatation_d

In [ ]:
train_dataset=train_dataset.shuffle(seed=42)
test_dataset=test_dataset.shuffle(seed=42)
val_dataset=val_dataset.shuffle(seed=42)

In [ ]:
bitsandbytes_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


In [ ]:
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
model=AutoModelForCausalLM.from_pretrained(model_name,quantization_config=bitsandbytes_config,device_map='auto',dtype=torch.bfloat16)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token=tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
model=prepare_model_for_kbit_training(model)

In [ ]:
lora_config=LoraConfig(r=16,lora_alpha=32,target_modules=['q_proj','v_proj'],
                       lora_dropout=0.05,bias='none',task_type='CAUSAL_LM')

In [ ]:
model=get_peft_model(model,lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [ ]:
training_args=SFTConfig(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    lr_scheduler_type='constant',
    logging_steps=10,
    dataset_text_field='text',
    max_length=512,
    optim="paged_adamw_8bit",
    bf16=True,
    fp16=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant":False}

)

In [ ]:
from transformers import EarlyStoppingCallback
trainer=SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.165096,1.194561,1.175158,1207661.000000,0.727634


TrainOutput(global_step=625, training_loss=1.2356903396606445, metrics={'train_runtime': 7294.4166, 'train_samples_per_second': 0.685, 'train_steps_per_second': 0.086, 'total_flos': 1.2381489415766016e+16, 'train_loss': 1.2356903396606445, 'epoch': 1.0})

In [ ]:
trainer.save_model("./content/docstring_final")
tokenizer.save_pretrained("./content/docstring_final")

('./content/docstring_final/tokenizer_config.json',
 './content/docstring_final/chat_template.jinja',
 './content/docstring_final/tokenizer.json')

In [ ]:
!zip -r docstring_final.zip /content/content/docstring_final

  adding: content/content/docstring_final/ (stored 0%)
  adding: content/content/docstring_final/training_args.bin (deflated 53%)
  adding: content/content/docstring_final/adapter_config.json (deflated 59%)
  adding: content/content/docstring_final/README.md (deflated 66%)
  adding: content/content/docstring_final/adapter_model.safetensors (deflated 53%)
  adding: content/content/docstring_final/tokenizer_config.json (deflated 46%)
  adding: content/content/docstring_final/tokenizer.json (deflated 85%)
  adding: content/content/docstring_final/chat_template.jinja (deflated 60%)


In [ ]:
from google.colab import files
files.download("docstring_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>